# Day 4 模块 4：交叉验证与网格搜索

Day 3 只固定了一组参数。今天学习：

1. 为什么只划分一次不够稳 → **K 折交叉验证**
2. **GridSearchCV**：在小网格里自动试参数


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day4_cafe_sales.csv'),
    Path('day4') / 'day4_cafe_sales.csv',
    Path('教学课程') / 'day4' / 'day4_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day4_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 准备特征 X 和目标 y（与 Day 3 同一套，方便对比）
# 特征列：可能影响营收的输入
feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp',
    'quality', 'competitors', 'holiday', 'location',
]
# 天气文字 → 数字分（晴最好，大雨最差）
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()  # 特征表
y = df['sales'].copy()  # 目标：营收

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.2：约 20% 当测试集；random_state=42：随机种子，结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


## 1. 交叉验证（K-Fold）在干什么

只做一次 `train_test_split`：运气好/坏都会影响分数。

**K 折**粗记：

```text
把训练数据切成 K 份
轮流：留下 1 份当验证，其余训练
得到 K 个分数 → 再平均
```

今天用 `cv=3`（三折）：快，够建立感觉。正式项目常见 5 折。

> 交叉验证用的是**训练集内部**轮换；最后仍可用一开始划出的 **X_test** 做最终检查。


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np

rf = RandomForestRegressor(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
)

# scoring='r2'：每一折算 R²；cv=3：三折
scores = cross_val_score(rf, X_train, y_train, cv=3, scoring='r2')
print('三折 R²:', np.round(scores, 3))
print('平均 R²:', round(scores.mean(), 3))
print('标准差:', round(scores.std(), 3), '（越小说明折与折之间越稳）')


## 2. GridSearchCV：小网格搜旋钮

思路：

```text
列出要试的超参数组合（网格不要太大）
对每一组做交叉验证
留下平均验证分最好的一组
再用最好参数，在全部训练数据上 fit
```

今天网格故意**很小**，保证课堂能跑完：


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [4, 8],
    'min_samples_leaf': [1, 4],
}

base = RandomForestRegressor(random_state=42, n_jobs=-1)
gs = GridSearchCV(
    base,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
)
gs.fit(X_train, y_train)

print('最佳参数:', gs.best_params_)
print('交叉验证最佳 R²:', round(gs.best_score_, 3))
print('测试集 R²（最终检查）:', round(gs.score(X_test, y_test), 3))


## 3. 和「手写固定参数」对比


In [ ]:
# Day 3 固定参数
rf_fixed = RandomForestRegressor(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
)
rf_fixed.fit(X_train, y_train)

print('固定参数 测试 R²:', round(rf_fixed.score(X_test, y_test), 3))
print('网格搜索 测试 R²:', round(gs.score(X_test, y_test), 3))
print('网格选中:', gs.best_params_)


## 4. 课堂注意

- 网格越大，组合数 = 各列表长度相乘，**越慢**
- 在**测试集上反复调参**会把测试集变成「第二训练集」→ 今天用 CV 在训练里选参，测试集尽量只做最后一眼
- 小数据上「搜完」和「固定 100/8」可能差不多：也正常，说明默认已经不差

## 5. 小练习

1. `n_estimators` 和 `max_depth` 哪个是超参数？叶子上的平均营收算参数还是超参数？
2. 三折分数一个很高、一个很低，说明什么？
3. 为什么不建议把网格开成几十上百组在课堂硬跑？


In [ ]:
# 在这里写
